## LAPD Crime Data Analysis

Los Angeles is one of the most populous cities in the US — and like any major city, it has a complex crime landscape. This notebook explores publicly available LAPD crime data to uncover patterns in when, where, and against whom crimes are committed.

The analysis focuses on three questions:
- Which hour of the day sees the highest volume of crime?
- Which area has the most night-time crime (10pm–3:59am)?
- Which victim age group is most frequently targeted?

## The data
### **crimes.csv**
A modified version of the original dataset, publicly available from Los Angeles Open Data.

| Column | Description |
|--------|-------------|
| `DR_NO` | Official file number (2-digit year + area ID + 5 digits) |
| `Date Rptd` | Date reported (MM/DD/YYYY) |
| `DATE OCC` | Date of occurrence (MM/DD/YYYY) |
| `TIME OCC` | Time of occurrence in 24-hour military time |
| `AREA NAME` | LAPD patrol division name |
| `Crm Cd Desc` | Crime description |
| `Vict Age` | Victim age in years |
| `Vict Sex` | Victim sex: F = Female, M = Male, X = Unknown |
| `Vict Descent` | Victim descent code (A–Z) |
| `Weapon Desc` | Weapon used, if applicable |
| `Status Desc` | Crime status |
| `LOCATION` | Street address of the crime |

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

crimes = pd.read_csv("crimes.csv", dtype={"TIME OCC": str})
crimes.head()

In [33]:
# Extract hour from TIME OCC and plot crime frequency by hour
crimes["HOUR OCC"] = crimes["TIME OCC"].str[:2].astype(int)

fig, ax = plt.subplots(figsize=(12, 5))
sns.countplot(data=crimes, x="HOUR OCC", ax=ax, color='steelblue')
ax.set_title("Crime Frequency by Hour of Day")
ax.set_xlabel("Hour (24h)")
ax.set_ylabel("Number of Crimes")
plt.tight_layout()
plt.show()

peak_crime_hour = crimes["HOUR OCC"].value_counts().idxmax()
print(f"Peak crime hour: {peak_crime_hour}:00")

In [34]:
# Night-time crime analysis (10pm–3:59am)
night_time = crimes[crimes["HOUR OCC"].isin([22, 23, 0, 1, 2, 3])]

night_by_area = (
    night_time.groupby("AREA NAME", as_index=False)["HOUR OCC"]
    .count()
    .rename(columns={"HOUR OCC": "crime_count"})
    .sort_values("crime_count", ascending=False)
)

peak_night_crime_location = night_by_area.iloc[0]["AREA NAME"]
print(f"Area with most night-time crime: {peak_night_crime_location}")
print(night_by_area.head(10).to_string(index=False))

In [35]:
# Victim age group analysis
age_bins = [0, 17, 25, 34, 44, 54, 64, np.inf]
age_labels = ["0-17", "18-25", "26-34", "35-44", "45-54", "55-64", "65+"]

crimes["Age Bracket"] = pd.cut(crimes["Vict Age"], bins=age_bins, labels=age_labels)

victim_ages = crimes["Age Bracket"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
victim_ages.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("Crimes by Victim Age Group")
ax.set_xlabel("Age Bracket")
ax.set_ylabel("Number of Crimes")
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print(victim_ages)